# 🏥 ViBioMIR - Kaggle High-Speed Retrieval & Indexing Pipeline

Notebook này được tối ưu riêng để chạy trực tiếp trên **Kaggle GPU (Tesla T4 / P100 / Dual T4)** nhằm:
1. **Cài đặt đầy đủ 100% các thư viện cần thiết** (FAISS GPU, FlagEmbedding BGE-M3, BM25, pyvi, jieba, reranker).
2. **Tải toàn bộ corpus Parquet mới** từ Hugging Face Storage Bucket `hf://buckets/nieths/ViBioMIR/corpus` (~351,674 tài liệu đã deduplicate).
3. **Tự động xây dựng Index mới (FAISS GPU Dense + BM25 Multilingual Sparse)**.
4. **(Tùy chọn) Lưu trữ Index mới lên HF Bucket** `hf://buckets/nieths/ViBioMIR/indices` để các lần inference sau không cần tốn thời gian build lại.
5. **Inference Hybrid Retrieval + Cross-Encoder Reranking** tối ưu theo độ đo Macro F2.
6. **Đóng gói file nộp bài** `submission.zip` trực tiếp tại `/kaggle/working/submission.zip` (1-click download).

## 1. Kiểm tra GPU và Cài đặt Đầy đủ Môi trường

In [16]:
# 1. Kiểm tra thông tin GPU
!nvidia-smi

Sat Oct  3 06:56:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
# 2. Cài đặt toàn bộ thư viện cần thiết cho Dense + Sparse Retrieval
!pip install -q --upgrade pip
!pip install -q faiss-gpu pyarrow "huggingface_hub[cli]" rank-bm25 pyvi jieba loguru typer pydantic pyyaml tqdm flagembedding sentence-transformers trafilatura httpx qdrant-client

# Đảm bảo CLI 'hf' được cài đặt
!which hf || pip install -q hf

/usr/local/bin/hf


## 2. Clone hoặc Cập nhật Codebase từ GitHub

In [17]:
import os
from pathlib import Path

REPO_URL = "https://github.com/neiths/med-doc-retrieval.git"
PROJECT_DIR = Path("/kaggle/working/med-doc-retrieval")

if not PROJECT_DIR.exists():
    print("Cloning repository...")
    !git clone -b feature/improve-data-quality $REPO_URL $PROJECT_DIR
else:
    print("Updating repository...")
    %cd $PROJECT_DIR
    !git pull origin feature/improve-data-quality

%cd $PROJECT_DIR
print("Working directory:", os.getcwd())

Updating repository...
/kaggle/working/med-doc-retrieval
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.13 KiB | 436.00 KiB/s, done.
From https://github.com/neiths/med-doc-retrieval
 * branch            feature/improve-data-quality -> FETCH_HEAD
   e708e48..db669b6  feature/improve-data-quality -> origin/feature/improve-data-quality
Updating ea4d958..db669b6
Fast-forward
 notebooks/04_kaggle_pipeline.ipynb | 68 ++++++++++++++++++++++++++++++++++----
 src/embedding/bge_m3.py            | 47 +++++++++++++++++++++-----
 src/pipeline.py                    | 24 ++++++++++----
 3 files changed, 117 insertions(+), 22 deletions(-)
/kaggle/working/med-doc-retrieval
Working directory: /kaggle/working/med-doc-retrieval


## 3. Cấu hình Hugging Face Token & Đồng bộ Corpus Parquet mới

Tải 4 shard Parquet (~351k bài) từ Storage Bucket `hf://buckets/nieths/ViBioMIR/corpus`:

In [18]:
import os
from huggingface_hub import login

# Điền Hugging Face Token của bạn (cần quyền Write nếu muốn đẩy index lên bucket)
HF_TOKEN = ""  # Nhập token hf_... của bạn ở đây
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)

# Đồng bộ 4 shard Parquet corpus mới từ bucket
!mkdir -p data/processed/parquet_corpus
!hf sync hf://buckets/nieths/ViBioMIR/corpus data/processed/parquet_corpus

# Tải sẵn cache bài báo PubMed (42MB) về local để inference 100% offline
!hf cp hf://buckets/nieths/ViBioMIR/processed/pubmed_cache.jsonl data/processed/pubmed_cache.jsonl || true

# Liệt kê các shard Parquet và file cache đã tải về
!ls -lh data/processed/parquet_corpus
!ls -lh data/processed/pubmed_cache.jsonl 2>/dev/null || true


Scanning remote bucket (4 files)0m
Scanning local directory (4 files)
Comparing files (4 paths)
Sync plan: hf://buckets/nieths/ViBioMIR/corpus -> data/processed/parquet_corpus
  Uploads: 0
  Downloads: 0
  Deletes: 0
  Skips: 4
Nothing to sync.

✓ Downloaded
  src: hf://buckets/nieths/ViBioMIR/processed/pubmed_cache.jsonl
  dst: data/processed/pubmed_cache.jsonl
total 453M
-rw-r--r-- 1 root root 142M Oct  3 05:55 corpus_part_000.parquet
-rw-r--r-- 1 root root 124M Oct  3 05:55 corpus_part_001.parquet
-rw-r--r-- 1 root root 126M Oct  3 05:55 corpus_part_002.parquet
-rw-r--r-- 1 root root  63M Oct  3 05:55 corpus_part_003.parquet
-rw-r--r-- 1 root root 42M Oct  3 06:57 data/processed/pubmed_cache.jsonl


## 4. Xây dựng Index mới (FAISS GPU + BM25) & Chạy Inference

Chạy toàn bộ pipeline từ đầu với data mới:
- Chunking tài liệu (chuẩn hóa câu, độ dài tối ưu 800 chars).
- Embed toàn bộ kho tài liệu bằng GPU BGE-M3 (FAISS GPU).
- Xây dựng index BM25 đa ngôn ngữ (pyvi cho tiếng Việt, jieba cho tiếng Trung).
- Truy vấn Hybrid RRF + Reranker `bge-reranker-large`.
- Tự động upload index mới lên HF Bucket với cờ `--sync-index-to-bucket` (tiết kiệm thời gian cho các lần chạy sau).
- Tự động xuất file nộp bài chuẩn tại `/kaggle/working/submission.zip`.

In [ ]:
# Chạy inference và build index mới từ parquet_corpus
!python scripts/colab_runner.py \
    --corpus-dir data/processed/parquet_corpus \
    --batch-size 64 \
    --output-name submission.json \
    --sync-index-to-bucket

## 5. Kiểm tra File Submission đã hoàn thành

In [ ]:
# Kiểm tra file zip nộp bài trong thư mục làm việc của Kaggle
!ls -lh /kaggle/working/*.zip
!unzip -l /kaggle/working/submission.zip | head -n 20